<h1 style="font-size: 48px; font-weight: bold;">
1. Peak Modeling
</h1>

<h2 style="font-weight: bold; font-size: 28px;">
1.1 Data Loading
</h2>

In [2]:
import pandas as pd

feat_28 = pd.read_csv("py_data/peak_feat_28.csv")
measurements = pd.read_csv("py_data/peak_measurements.csv")

<h2 style="font-weight: bold; font-size: 28px;">
1.2 Name Mapping
</h2>

In [3]:
name_map = {
    "slope_log": "log_slope",
    "slope_y_1_7": "short_slopes",
    "longest_inc_run": "inc_run",
    "zero_count": "nonzero_days",

    "c22_DN_HistogramMode_5": "mode_5",
    "c22_DN_HistogramMode_10": "mode_10",
    "c22_CO_f1ecac": "acf_timescale",
    "c22_CO_FirstMin_ac": "acf_first_min",
    "c22_CO_HistogramAMI_even_2_5": "ami2",
    "c22_CO_trev_1_num": "trev",
    "c22_MD_hrv_classic_pnn40": "high_fluctuation",
    "c22_SB_BinaryStats_mean_longstretch1": "stretch_high",
    "c22_SB_TransitionMatrix_3ac_sumdiagcov": "transition_variance",
    "c22_PD_PeriodicityWang_th0_01": "periodicity",
    "c22_CO_Embed2_Dist_tau_d_expfit_meandiff": "embedding_dist",
    "c22_IN_AutoMutualInfoStats_40_gaussian_fmmi": "ami_timescale",
    "c22_FC_LocalSimple_mean1_tauresrat": "whiten_timescale",
    "c22_DN_OutlierInclude_p_001_mdrmd": "outlier_timing_pos",
    "c22_DN_OutlierInclude_n_001_mdrmd": "outlier_timing_neg",
    "c22_SP_Summaries_welch_rect_area_5_1": "low_freq_power",
    "c22_SB_BinaryStats_diff_longstretch0": "stretch_decreasing",
    "c22_SB_MotifThree_quantile_hh": "entropy_pairs",
    "c22_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1": "rs_range",
    "c22_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1": "dfa",
    "c22_SP_Summaries_welch_rect_centroid": "centroid_freq",
    "c22_FC_LocalSimple_mean3_stderr": "forecast_error"
}

feat_28 = feat_28.rename(columns={k: v for k, v in name_map.items() if k in feat_28.columns})
feat_28.columns

Index(['country', 'lineage', 'log_slope', 'short_slopes', 'peak_val',
       'auc_norm', 'inc_run', 'nonzero_days', 'mode_5', 'mode_10',
       'acf_timescale', 'acf_first_min', 'ami2', 'trev', 'high_fluctuation',
       'stretch_high', 'transition_variance', 'periodicity', 'embedding_dist',
       'ami_timescale', 'whiten_timescale', 'outlier_timing_pos',
       'outlier_timing_neg', 'low_freq_power', 'stretch_decreasing',
       'entropy_pairs', 'rs_range', 'dfa', 'centroid_freq', 'forecast_error'],
      dtype='object')

<h2 style="font-weight: bold; font-size: 28px;">
1.3 Data Merging
</h2>

In [ ]:
df = feat_28.drop(columns=["auc_norm", "peak_val"], errors="ignore").merge(
    measurements[["country", "lineage", "peak_share", "peak_share_logit", "peak_share_cat"]],
    on=["country", "lineage"],
    how="left"
)

df = df.dropna(subset=["peak_share_cat"])
peak_class_order = ["<0.10", "0.10–0.20", ">0.20"]
df["peak_share_cat"] = pd.Categorical(
    df["peak_share_cat"],
    categories=peak_class_order,
    ordered=True
)


<h2 style="font-weight: bold; font-size: 28px;">
1.4 Modeling Training
</h2>

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

SHAP_SEED = 2025
SHAP_CUTOFF = 0.5
SHAP_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 4,
    "min_child_weight": 3,
    "gamma": 0.5,
    "lambda": 2.0,
    "alpha": 1.0,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": SHAP_SEED,
}


def fit_binary_xgb(X, y, params=None, num_boost_round=500):
    dtrain = xgb.DMatrix(X, label=y, feature_names=list(X.columns))
    return xgb.train(
        params or SHAP_XGB_PARAMS,
        dtrain,
        num_boost_round=num_boost_round,
        verbose_eval=False
    )


# Match code/04_peak_modeling.R:
#   X drops peak_share, peak_share_logit, and peak_share_cat;
#   binary1 is "<0.10" vs ">0.10";
#   binary2 is "<0.20" vs ">0.20";
#   the final three-class label is reconstructed from the two binary predictions.
X = df.drop(
    columns=["country", "lineage", "peak_share", "peak_share_logit", "peak_share_cat"],
    errors="ignore"
)
X_train = X

peak_y_binary1 = np.where(df["peak_share_cat"].astype(str) == "<0.10", 0, 1)
peak_y_binary2 = np.where(df["peak_share_cat"].astype(str) == ">0.20", 1, 0)

peak_bst_binary1 = fit_binary_xgb(X_train, peak_y_binary1)
peak_bst_binary2 = fit_binary_xgb(X_train, peak_y_binary2)

peak_prob_binary1 = peak_bst_binary1.predict(xgb.DMatrix(X_train, feature_names=list(X_train.columns)))
peak_prob_binary2 = peak_bst_binary2.predict(xgb.DMatrix(X_train, feature_names=list(X_train.columns)))

peak_pred_binary1 = np.where(peak_prob_binary1 > SHAP_CUTOFF, ">0.10", "<0.10")
peak_pred_binary2 = np.where(peak_prob_binary2 > SHAP_CUTOFF, ">0.20", "<0.20")
peak_pred = np.select(
    [
        peak_pred_binary1 == "<0.10",
        (peak_pred_binary1 == ">0.10") & (peak_pred_binary2 == "<0.20"),
        (peak_pred_binary1 == ">0.10") & (peak_pred_binary2 == ">0.20"),
    ],
    ["<0.10", "0.10–0.20", ">0.20"],
    default=np.nan
)

print("Classification Report:")
print(classification_report(df["peak_share_cat"].astype(str), peak_pred, labels=peak_class_order))

print("Confusion Matrix:")
print(confusion_matrix(df["peak_share_cat"].astype(str), peak_pred, labels=peak_class_order))

# SHAP figures focus on the high-peak binary model, matching the second binary task in 04_peak_modeling.R.
bst = peak_bst_binary2
shap_target = "peak_share_cat >0.20"


<h2 style="font-weight: bold; font-size: 28px;">
1.5 SHAP Analysis I
</h2>

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.Explainer(bst, X_train)
shap_values = explainer(X_train)


In [ ]:
plt.figure(figsize=(10, 8))

# important: show=False makes it draw on the matplotlib figure
shap.plots.beeswarm(
    shap_values,
    max_display=15,
    show=False
)

plt.tight_layout()
plt.savefig(
    "../result/figs/shap_beeswarm_peak.tiff",
    dpi=600,
    format="tiff",
    pil_kwargs={"compression": "tiff_lzw"}
)
plt.close()


In [ ]:
plt.figure(figsize=(10, 8))

# important: show=False makes it draw on the matplotlib figure
shap.plots.bar(
    shap_values,
    max_display=15,
    show=False
)

plt.tight_layout()
plt.savefig(
    "../result/figs/shap_values_peak.tiff",
    dpi=600,
    format="tiff",
    pil_kwargs={"compression": "tiff_lzw"}
)
plt.close()


<h1 style="font-size: 48px; font-weight: bold;">
2. Duration Modeling
</h1>

<h2 style="font-weight: bold; font-size: 28px;">
2.1 Data Loading
</h2>

In [20]:
feat_21 = pd.read_csv("py_data/duration_feat_21.csv")
measurements = pd.read_csv("py_data/duration_measurements.csv")

In [21]:
feat_21 = feat_21.rename(columns={k: v for k, v in name_map.items() if k in feat_21.columns})
feat_21.columns

Index(['country', 'lineage', 'log_slope', 'short_slopes', 'peak_val',
       'auc_norm', 'inc_run', 'nonzero_days', 'mode_5', 'mode_10',
       'acf_timescale', 'acf_first_min', 'ami2', 'trev', 'high_fluctuation',
       'stretch_high', 'transition_variance', 'periodicity', 'embedding_dist',
       'ami_timescale', 'whiten_timescale', 'outlier_timing_pos',
       'outlier_timing_neg', 'low_freq_power', 'stretch_decreasing',
       'entropy_pairs', 'rs_range', 'dfa', 'centroid_freq', 'forecast_error'],
      dtype='object')

<h2 style="font-weight: bold; font-size: 28px;">
2.2 Data Merging
</h2>

In [ ]:
df = feat_21.drop(columns=["auc_norm", "peak_val"], errors="ignore").merge(
    measurements[["country", "lineage", "days_above_10", "days_above_10_cat"]],
    on=["country", "lineage"],
    how="left"
)

df["days_above_10_cat"] = df["days_above_10_cat"].replace({"0": "<30", "1–30": "<30"})
df = df.dropna(subset=["days_above_10_cat"])
duration_class_order = ["<30", "31–100", "100+"]
df["days_above_10_cat"] = pd.Categorical(
    df["days_above_10_cat"],
    categories=duration_class_order,
    ordered=True
)


<h2 style="font-weight: bold; font-size: 28px;">
2.3 Modeling Training
</h2>

In [ ]:
# Match code/04_duration_modeling.R:
#   X drops days_above_10 and days_above_10_cat;
#   binary1 is "<30" vs "≥30";
#   binary2 is "<100" vs "100+";
#   the final three-class label is reconstructed from the two binary predictions.
X = df.drop(
    columns=["country", "lineage", "days_above_10", "days_above_10_cat"],
    errors="ignore"
)
X_train = X

duration_y_binary1 = np.where(df["days_above_10_cat"].astype(str) == "<30", 0, 1)
duration_y_binary2 = np.where(df["days_above_10_cat"].astype(str) == "100+", 1, 0)

duration_bst_binary1 = fit_binary_xgb(X_train, duration_y_binary1)
duration_bst_binary2 = fit_binary_xgb(X_train, duration_y_binary2)

duration_prob_binary1 = duration_bst_binary1.predict(xgb.DMatrix(X_train, feature_names=list(X_train.columns)))
duration_prob_binary2 = duration_bst_binary2.predict(xgb.DMatrix(X_train, feature_names=list(X_train.columns)))

duration_pred_binary1 = np.where(duration_prob_binary1 > SHAP_CUTOFF, "≥30", "<30")
duration_pred_binary2 = np.where(duration_prob_binary2 > SHAP_CUTOFF, "100+", "<100")
duration_pred = np.select(
    [
        duration_pred_binary1 == "<30",
        (duration_pred_binary1 == "≥30") & (duration_pred_binary2 == "<100"),
        (duration_pred_binary1 == "≥30") & (duration_pred_binary2 == "100+"),
    ],
    ["<30", "31–100", "100+"],
    default=np.nan
)

print("Classification Report:")
print(classification_report(df["days_above_10_cat"].astype(str), duration_pred, labels=duration_class_order))

print("Confusion Matrix:")
print(confusion_matrix(df["days_above_10_cat"].astype(str), duration_pred, labels=duration_class_order))

# SHAP figures focus on the long-duration binary model, matching the second binary task in 04_duration_modeling.R.
bst = duration_bst_binary2
shap_target = "days_above_10_cat 100+"


<h2 style="font-weight: bold; font-size: 28px;">
2.4 SHAP Analysis II
</h2>

In [ ]:
explainer = shap.Explainer(bst, X_train)
shap_values = explainer(X_train)


In [ ]:
plt.figure(figsize=(10, 8))

# important: show=False makes it draw on the matplotlib figure
shap.plots.bar(
    shap_values,
    max_display=15,
    show=False
)

plt.tight_layout()
plt.savefig(
    "../result/figs/shap_values_duration.tiff",
    dpi=600,
    format="tiff",
    pil_kwargs={"compression": "tiff_lzw"}
)
plt.close()


In [ ]:
plt.figure(figsize=(10, 8))

# important: show=False makes it draw on the matplotlib figure
shap.plots.beeswarm(
    shap_values,
    max_display=15,
    show=False
)

plt.tight_layout()
plt.savefig(
    "../result/figs/shap_beeswarm_duration.tiff",
    dpi=600,
    format="tiff",
    pil_kwargs={"compression": "tiff_lzw"}
)
plt.close()
